# Task 3: Data Cleaning and Text Preprocessing

## 1. Objective

The objective of this task is to prepare the TED Talks datasets for a content-based recommendation system.

The original datasets will be loaded from `data/raw/`. Missing values, duplicate records, and text-formatting issues will be investigated and handled appropriately.

The main dataset and transcript dataset will be matched using the `url` column. A new `combined_text` column will be created using the cleaned title, tags, description, and transcript.

The cleaned dataset will be saved to `data/processed/ted_talks_cleaned.csv` without modifying the original CSV files.

### Preprocessing decisions

- Preserve all valid TED Talk records.
- Replace missing speaker occupations with `"Unknown"`.
- Remove confirmed exact duplicate transcript rows.
- Investigate duplicate transcript URLs before merging.
- Preserve valid numeric outliers.
- Normalize text and whitespace while retaining meaningful words.
- Keep original text fields and create separate cleaned text fields.
- Use a left join to retain all talks from the main dataset.

In [54]:
import pandas as pd
import numpy as np
import re
import ast
from pathlib import Path

In [55]:
df1 = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\raw\ted_talks.csv")
df2 = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\raw\ted_transcripts.csv")

print("Main dataset shape:", df1.shape)
print("Transcript dataset shape:", df2.shape)

Main dataset shape: (2550, 17)
Transcript dataset shape: (2467, 2)


In [56]:
df1.head(2)

,comments,description,duration,event,film_date,languages,main_speaker,name,num_speaker,published_date,ratings,related_talks,speaker_occupation,tags,title,url,views
0,4553,Sir Ken Robinson makes an entertaining and pro...,1164,TED2006,1140825600,60,Ken Robinson,Ken Robinson: Do schools kill creativity?,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 19645}, {...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/im...",Author/educator,"['children', 'creativity', 'culture', 'dance',...",Do schools kill creativity?,https://www.ted.com/talks/ken_robinson_says_sc...,47227110
1,265,With the same humor and humanity he exuded in ...,977,TED2006,1140825600,43,Al Gore,Al Gore: Averting the climate crisis,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 544}, {'i...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/im...",Climate advocate,"['alternative energy', 'cars', 'climate change...",Averting the climate crisis,https://www.ted.com/talks/al_gore_on_averting_...,3200520


In [57]:
df2.head(2)

,transcript,url
0,Good morning. How are you?(Laughter)It's been ...,https://www.ted.com/talks/ken_robinson_says_sc...
1,"Thank you so much, Chris. And it's truly a gre...",https://www.ted.com/talks/al_gore_on_averting_...


In [58]:
main_raw = df1.copy(deep=True)
transcript_raw = df2.copy(deep=True)

main_clean = main_raw.copy(deep=True)
transcript_clean = transcript_raw.copy(deep=True)

In [59]:
print("Main dataset missing values:")
print(main_clean.isnull().sum())

print("Main dataset exact duplicate rows:",main_clean.duplicated().sum())

print("Transcript dataset missing values:")
display(transcript_clean.isnull().sum())

print("Transcript exact duplicate rows:",transcript_clean.duplicated().sum())

print("Main dataset duplicate URLs:",main_clean["url"].duplicated().sum())

print("Transcript duplicate URLs:",transcript_clean["url"].duplicated().sum())

Main dataset missing values:
comments              0
description           0
duration              0
event                 0
film_date             0
languages             0
main_speaker          0
name                  0
num_speaker           0
published_date        0
ratings               0
related_talks         0
speaker_occupation    6
tags                  0
title                 0
url                   0
views                 0
dtype: int64
Main dataset exact duplicate rows: 0
Transcript dataset missing values:


transcript    0
url           0
dtype: int64

Transcript exact duplicate rows: 3
Main dataset duplicate URLs: 0
Transcript duplicate URLs: 3


In [60]:
def inspect_text_values(data, columns):
    results = []

    for col in columns:
        if col in data.columns:
            text = data[col].astype("string")

            results.append({
                "Column": col,
                "Missing_Values": data[col].isna().sum(),
                "Empty_Or_Whitespace": (
                    text.str.strip().eq("").fillna(False).sum()
                )
            })

    return pd.DataFrame(results)


print("Main dataset text quality:")
display(
    inspect_text_values(
        main_clean,
        ["title", "description", "tags", "speaker_occupation", "url"]
    )
)

print("Transcript dataset text quality:")
display(
    inspect_text_values(
        transcript_clean,
        ["transcript", "url"]
    )
)

Main dataset text quality:


,Column,Missing_Values,Empty_Or_Whitespace
0,title,0,0
1,description,0,0
2,tags,0,0
3,speaker_occupation,6,0
4,url,0,0


Transcript dataset text quality:


,Column,Missing_Values,Empty_Or_Whitespace
0,transcript,0,0
1,url,0,0


In [61]:
# Imputing speaker_occupation missing values with word named Unkown
main_clean["speaker_occupation"] = main_clean["speaker_occupation"].fillna("Unknown")

In [62]:
print("Missing occupations after impuating:\n",main_clean.isnull().sum())

Missing occupations after impuating:
 comments              0
description           0
duration              0
event                 0
film_date             0
languages             0
main_speaker          0
name                  0
num_speaker           0
published_date        0
ratings               0
related_talks         0
speaker_occupation    0
tags                  0
title                 0
url                   0
views                 0
dtype: int64


In [63]:
print("In Main dataset duplicates identified:", main_clean.duplicated().sum())
print("In Transcript dataset duplicates identified:",transcript_clean.duplicated().sum())

In Main dataset duplicates identified: 0
In Transcript dataset duplicates identified: 3


In [64]:
duplicate_transcripts = transcript_clean[transcript_clean["url"].duplicated(keep=False)].sort_values("url")

display(duplicate_transcripts[["url", "transcript"]])

,url,transcript
1109,https://www.ted.com/talks/brene_brown_listenin...,I'm going to tell you a little bit about my TE...
1116,https://www.ted.com/talks/brene_brown_listenin...,I'm going to tell you a little bit about my TE...
1107,https://www.ted.com/talks/jonathan_haidt_human...,I have a question for you: Are you religious? ...
1114,https://www.ted.com/talks/jonathan_haidt_human...,I have a question for you: Are you religious? ...
1108,https://www.ted.com/talks/rob_reid_the_8_billi...,The recent debate over copyright laws like SOP...
1115,https://www.ted.com/talks/rob_reid_the_8_billi...,The recent debate over copyright laws like SOP...


In [65]:
transcript_clean = transcript_clean.drop_duplicates()

In [66]:
print(f"After removing duplicates the transcript dataset has {transcript_clean.duplicated().sum()}* duplicate rows")

After removing duplicates the transcript dataset has 0* duplicate rows


In [67]:
main_clean.shape

(2550, 17)

In [68]:
transcript_clean.shape

(2464, 2)

In [69]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove stage directions such as (Laughter), (Applause), (Music)
    text = re.sub(r"\([^)]*\)", " ", text)

    # Remove square-bracketed stage directions, if present
    text = re.sub(r"\[[^\]]*\]", " ", text)

    # Add a space after sentence-ending punctuation when missing
    text = re.sub(r"([.!?])(?=[A-Za-z])", r"\1 ", text)

    # Remove unnecessary leading, trailing, and repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Convert text to lowercase
    text = text.lower()

    return text

In [70]:
main_text_columns = ["title", "description"]

for col in main_text_columns:
    main_clean[col] = main_clean[col].apply(clean_text)

display(main_clean[["title", "description"]].head())

,title,description
0,do schools kill creativity?,sir ken robinson makes an entertaining and pro...
1,averting the climate crisis,with the same humor and humanity he exuded in ...
2,simplicity sells,new york times columnist david pogue takes aim...
3,greening the ghetto,"in an emotionally charged talk, macarthur-winn..."
4,the best stats you've ever seen,you've never seen data presented like this. wi...


In [71]:
def parse_tags(value):
    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(str(value))

        if isinstance(parsed, list):
            return parsed

        return [str(parsed)]

    except (ValueError, SyntaxError):
        return [str(value)]


def clean_tags(value):
    tags = parse_tags(value)

    cleaned_tags = [
        clean_text(tag)
        for tag in tags
        if str(tag).strip()
    ]

    return " ".join(cleaned_tags)


main_clean["tags"] = main_clean["tags"].apply(clean_tags)

display(main_clean[["title", "tags"]].head(10))

,title,tags
0,do schools kill creativity?,children creativity culture dance education pa...
1,averting the climate crisis,alternative energy cars climate change culture...
2,simplicity sells,computers entertainment interface design media...
3,greening the ghetto,macarthur grant activism business cities envir...
4,the best stats you've ever seen,africa asia google demo economics global devel...
5,why we do what we do,business culture entertainment goal-setting mo...
6,letting go of god,christianity god atheism comedy culture humor ...
7,behind the design of seattle's library,architecture collaboration culture design library
8,let's teach religion -- all religion -- in sch...,god ted brain trust atheism brain cognitive sc...
9,a life of purpose,christianity god culture happiness leadership ...


In [72]:
transcript_clean["transcript"] = (
    transcript_clean["transcript"].apply(clean_text)
)

display(transcript_clean[["url", "transcript"]].head(3))

,url,transcript
0,https://www.ted.com/talks/ken_robinson_says_sc...,"good morning. how are you? it's been great, ha..."
1,https://www.ted.com/talks/al_gore_on_averting_...,"thank you so much, chris. and it's truly a gre..."
2,https://www.ted.com/talks/david_pogue_says_sim...,"hello voice mail, my old friend. i've called f..."


In [73]:
sample_transcript = transcript_clean["transcript"].iloc[0]

print(sample_transcript[:2000])

good morning. how are you? it's been great, hasn't it? i've been blown away by the whole thing. in fact, i'm leaving. there have been three themes running through the conference which are relevant to what i want to talk about. one is the extraordinary evidence of human creativity in all of the presentations that we've had and in all of the people here. just the variety of it and the range of it. the second is that it's put us in a place where we have no idea what's going to happen, in terms of the future. no idea how this may play out. i have an interest in education. actually, what i find is everybody has an interest in education. don't you? i find this very interesting. if you're at a dinner party, and you say you work in education — actually, you're not often at dinner parties, frankly. if you work in education, you're not asked. and you're never asked back, curiously. that's strange to me. but if you are, and you say to somebody, you know, they say, "what do you do?" and you say yo

In [74]:
# Numerical columns to check for outliers
numeric_columns = [
    "views",
    "comments",
    "duration",
    "languages",
    "num_speaker"
]

outlier_summary = []

for column in numeric_columns:
    Q1 = main_clean[column].quantile(0.25)
    Q3 = main_clean[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = main_clean[
        (main_clean[column] < lower_bound) |
        (main_clean[column] > upper_bound)
    ]

    outlier_summary.append({
        "Column": column,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers),
        "Outlier Percentage": round(
            len(outliers) / len(main_clean) * 100, 2
        )
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(outlier_summary)

,Column,Lower Bound,Upper Bound,Outlier Count,Outlier Percentage
0,views,-661657.375,3118209.625,246,9.65
1,comments,-175.125,459.875,200,7.84
2,duration,-127.625,1751.375,28,1.10
3,languages,8.000,48.000,164,6.43
4,num_speaker,1.000,1.000,58,2.27


In [75]:
# Inspect talks with unusually high views
Q1 = main_clean["views"].quantile(0.25)
Q3 = main_clean["views"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

high_view_talks = main_clean[
    main_clean["views"] > upper_bound
][["title", "main_speaker", "views", "url"]]

display(
    high_view_talks.sort_values(
        by="views",
        ascending=False
    ).head(10)
)

,title,main_speaker,views,url
0,do schools kill creativity?,Ken Robinson,47227110,https://www.ted.com/talks/ken_robinson_says_sc...
1346,your body language may shape who you are,Amy Cuddy,43155405,https://www.ted.com/talks/amy_cuddy_your_body_...
677,how great leaders inspire action,Simon Sinek,34309432,https://www.ted.com/talks/simon_sinek_how_grea...
837,the power of vulnerability,Brené Brown,31168150,https://www.ted.com/talks/brene_brown_on_vulne...
452,10 things you didn't know about orgasm,Mary Roach,22270883,https://www.ted.com/talks/mary_roach_10_things...
1776,how to speak so that people want to listen,Julian Treasure,21594632,https://www.ted.com/talks/julian_treasure_how_...
201,my stroke of insight,Jill Bolte Taylor,21190883,https://www.ted.com/talks/jill_bolte_taylor_s_...
5,why we do what we do,Tony Robbins,20685401,https://www.ted.com/talks/tony_robbins_asks_wh...
2114,this is what happens when you reply to spam email,James Veitch,20475972,https://www.ted.com/talks/james_veitch_this_is...
1416,"looks aren't everything. believe me, i'm a model.",Cameron Russell,19787465,https://www.ted.com/talks/cameron_russell_look...


In [76]:
# Check for negative views
invalid_views = main_clean[main_clean["views"] < 0]

print("Number of talks with negative views:", len(invalid_views))

display(invalid_views[["title", "views", "url"]])

# Keep all valid view outliers
main_clean = main_clean[main_clean["views"] >= 0].copy()

print("Dataset shape after checking views:", main_clean.shape)

Number of talks with negative views: 0


,title,views,url


Dataset shape after checking views: (2550, 17)


In [77]:
# Check for negative comments
invalid_comments = main_clean[main_clean["comments"] < 0]

print("Number of talks with negative comments:", len(invalid_comments))

display(invalid_comments[["title", "comments", "url"]])

# Keep all valid comment outliers
main_clean = main_clean[main_clean["comments"] >= 0].copy()

print("Dataset shape after checking comments:", main_clean.shape)

Number of talks with negative comments: 0


,title,comments,url


Dataset shape after checking comments: (2550, 17)


In [78]:
# Find zero or negative durations
invalid_duration = main_clean[main_clean["duration"] <= 0]

print("Number of talks with zero or negative duration:", len(invalid_duration))

display(invalid_duration[["title", "duration", "url"]])

# Keep only talks with positive duration
main_clean = main_clean[main_clean["duration"] > 0].copy()

print("Dataset shape after checking duration:", main_clean.shape)

Number of talks with zero or negative duration: 0


,title,duration,url


Dataset shape after checking duration: (2550, 17)


In [79]:
# Find negative language counts
invalid_languages = main_clean[main_clean["languages"] < 0]

print("Number of talks with negative language counts:", len(invalid_languages))

display(invalid_languages[["title", "languages", "url"]])

# Keep all valid language-count outliers
main_clean = main_clean[main_clean["languages"] >= 0].copy()

print("Dataset shape after checking languages:", main_clean.shape)

Number of talks with negative language counts: 0


,title,languages,url


Dataset shape after checking languages: (2550, 17)


In [80]:
# Find zero or negative speaker counts
invalid_speakers = main_clean[main_clean["num_speaker"] <= 0]

print("Number of talks with invalid speaker counts:", len(invalid_speakers))

display(invalid_speakers[["title", "num_speaker", "url"]])

# Keep only talks with at least one speaker
main_clean = main_clean[main_clean["num_speaker"] > 0].copy()

print("Dataset shape after checking num_speaker:", main_clean.shape)

Number of talks with invalid speaker counts: 0


,title,num_speaker,url


Dataset shape after checking num_speaker: (2550, 17)


In [81]:
print("Original number of talks:", len(main_raw))
print("Number of talks after outlier validation:", len(main_clean))
print("Number of talks removed:", len(main_raw) - len(main_clean))

display(
    main_clean[
        ["views", "comments", "duration", "languages", "num_speaker"]
    ].describe()
)

Original number of talks: 2550
Number of talks after outlier validation: 2550
Number of talks removed: 0


,views,comments,duration,languages,num_speaker
count,2.550000e+03,2550.000000,2550.000000,2550.000000,2550.000000
mean,1.698297e+06,191.562353,826.510196,27.326275,1.028235
std,2.498479e+06,282.315223,374.009138,9.563452,0.207705
min,5.044300e+04,2.000000,135.000000,0.000000,1.000000
25%,7.557928e+05,63.000000,577.000000,23.000000,1.000000
50%,1.124524e+06,118.000000,848.000000,28.000000,1.000000
75%,1.700760e+06,221.750000,1046.750000,33.000000,1.000000
max,4.722711e+07,6404.000000,5256.000000,72.000000,5.000000


In [82]:
# Select the required columns from the main dataset
main_columns = [
    "title",
    "description",
    "tags",
    "main_speaker",
    "speaker_occupation",
    "event",
    "views",
    "url"]

main_selected = main_clean[main_columns].copy()

# Select the URL and transcript columns
transcript_selected = transcript_clean[["url", "transcript"]].copy()

# Merge both datasets using URL
merged_df = pd.merge(main_selected,transcript_selected,on="url",how="left",validate="one_to_one")

# Display the results
print("Main dataset shape:", main_selected.shape)
print("Transcript dataset shape:", transcript_selected.shape)
print("Merged dataset shape:", merged_df.shape)

display(merged_df.head())

Main dataset shape: (2550, 8)
Transcript dataset shape: (2464, 2)
Merged dataset shape: (2550, 9)


,title,description,tags,main_speaker,speaker_occupation,event,views,url,transcript
0,do schools kill creativity?,sir ken robinson makes an entertaining and pro...,children creativity culture dance education pa...,Ken Robinson,Author/educator,TED2006,47227110,https://www.ted.com/talks/ken_robinson_says_sc...,"good morning. how are you? it's been great, ha..."
1,averting the climate crisis,with the same humor and humanity he exuded in ...,alternative energy cars climate change culture...,Al Gore,Climate advocate,TED2006,3200520,https://www.ted.com/talks/al_gore_on_averting_...,"thank you so much, chris. and it's truly a gre..."
2,simplicity sells,new york times columnist david pogue takes aim...,computers entertainment interface design media...,David Pogue,Technology columnist,TED2006,1636292,https://www.ted.com/talks/david_pogue_says_sim...,"hello voice mail, my old friend. i've called f..."
3,greening the ghetto,"in an emotionally charged talk, macarthur-winn...",macarthur grant activism business cities envir...,Majora Carter,Activist for environmental justice,TED2006,1697550,https://www.ted.com/talks/majora_carter_s_tale...,if you're here today — and i'm very happy that...
4,the best stats you've ever seen,you've never seen data presented like this. wi...,africa asia google demo economics global devel...,Hans Rosling,Global health expert; data visionary,TED2006,12005869,https://www.ted.com/talks/hans_rosling_shows_t...,"about 10 years ago, i took on the task to teac..."


In [83]:
# Check whether all main dataset records were preserved
print("Rows before merging:", len(main_selected))
print("Rows after merging:", len(merged_df))

# Count missing transcripts after merging
print(
    "Talks without transcripts:",
    merged_df["transcript"].isna().sum()
)

# Check for duplicate URLs
print(
    "Duplicate URLs in merged dataset:",
    merged_df["url"].duplicated().sum()
)

# Check the final columns
print("Final columns:", merged_df.columns.tolist())

Rows before merging: 2550
Rows after merging: 2550
Talks without transcripts: 86
Duplicate URLs in merged dataset: 0
Final columns: ['title', 'description', 'tags', 'main_speaker', 'speaker_occupation', 'event', 'views', 'url', 'transcript']


In [84]:
from pathlib import Path

# Create the processed folder if it does not exist
processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

# Save the cleaned and merged dataset
output_path = processed_folder / "ted_talks_cleaned.csv"

merged_df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File location:", output_path.resolve())
print("Dataset shape:", merged_df.shape)

Cleaned dataset saved successfully!
File location: C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\processed\ted_talks_cleaned.csv
Dataset shape: (2550, 9)
